# Kaggle: EagleVision Phase 1 Setup and Train

This notebook is for a real Kaggle session where you want to:

- clone `alooboii/EagleVision`
- install the repo in the notebook runtime
- normalize an attached indoor RGB-D dataset into the ScanNet-style layout expected by Phase 1
- download a Depth Anything V2 checkpoint
- generate Kaggle-local configs
- run a first Phase 1 training pass and validation eval

Expected Kaggle setup:

- internet enabled for cloning the repo and downloading checkpoints
- a dataset attached under `/kaggle/input/...`
- the attached dataset should either already contain ScanNet-style scene folders or be close enough to map into:
  - `scene_id/color/*.jpg`
  - `scene_id/depth/*.png`
  - `scene_id/pose/*.txt`


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import yaml

KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')
REPO_DIR = KAGGLE_WORKING / 'EagleVision'
REPO_URL = 'https://github.com/alooboii/EagleVision.git'

print('input exists:', KAGGLE_INPUT.exists())
print('working exists:', KAGGLE_WORKING.exists())
sorted([p.name for p in KAGGLE_INPUT.iterdir()])[:20] if KAGGLE_INPUT.exists() else []

## Clone The Repository And Install It

If the repo is already present in the session, the cell updates it. Otherwise it clones fresh.

In [ ]:
def run(cmd: list[str], cwd: Path | None = None, capture: bool = False):
    print('$', ' '.join(cmd))
    completed = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        check=True,
        text=True,
        capture_output=capture,
    )
    if capture:
        if completed.stdout:
            print(completed.stdout)
        if completed.stderr:
            print(completed.stderr)
        return completed
    return None

if REPO_DIR.exists():
    run(['git', 'fetch', '--all'], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)
else:
    run(['git', 'clone', REPO_URL, str(REPO_DIR)])

run(['python', '-m', 'pip', 'install', '-q', '-e', '.[dev]'], cwd=REPO_DIR)
print('repo ready:', REPO_DIR)


## Attach And Inspect Your Dataset

Recommended Kaggle dataset for this repository:

- `klein2111/scannet-2d`
- Kaggle link: `https://www.kaggle.com/datasets/klein2111/scannet-2d`
- default Kaggle mount path: `/kaggle/input/scannet-2d`

If you attach a differently named dataset or your own private ScanNet subset, update `RAW_DATASET_DIR` in the next cell.


In [ ]:
RAW_DATASET_DIR = KAGGLE_INPUT / 'scannet-2d'
RAW_DATASET_DIR

In [ ]:
if not RAW_DATASET_DIR.exists():
    raise FileNotFoundError(
        f'Attached dataset path not found: {RAW_DATASET_DIR}. '
        'Attach `klein2111/scannet-2d` in Kaggle Inputs or edit RAW_DATASET_DIR to match your dataset name.'
    )

preview = []
for idx, path in enumerate(sorted(RAW_DATASET_DIR.rglob('*'))):
    preview.append(str(path))
    if idx >= 30:
        break
preview

## Normalize The Dataset Into `data/scannet`

This cell supports the simplest and most reliable Kaggle workflow:

- if the attached dataset already contains ScanNet-style scene folders, it symlinks or copies them into `data/scannet`
- if your folder naming is different, edit the helper function below once and rerun

A valid scene must end up with:

- `scene_id/color/*.jpg`
- `scene_id/depth/*.png`
- `scene_id/pose/*.txt`


In [ ]:
import zipfile

TARGET_DATA_ROOT = REPO_DIR / 'data' / 'scannet'
TARGET_DATA_ROOT.mkdir(parents=True, exist_ok=True)
EXTRACT_ROOT = KAGGLE_WORKING / 'scannet_extracted'
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

def discover_scene_dirs(raw_root: Path) -> list[Path]:
    scenes = []
    for candidate in sorted(raw_root.rglob('*')):
        if not candidate.is_dir():
            continue
        if (candidate / 'color').exists() and (candidate / 'depth').exists() and (candidate / 'pose').exists():
            scenes.append(candidate)
    deduped = []
    seen = set()
    for scene in scenes:
        if scene.name not in seen:
            deduped.append(scene)
            seen.add(scene.name)
    return deduped

def extract_archives_if_needed(raw_root: Path, extract_root: Path) -> None:
    archives = sorted(list(raw_root.glob('*.zip')) + list(raw_root.glob('*.tar.gz')) + list(raw_root.glob('*.tgz')))
    for archive in archives:
        target_dir = extract_root / archive.stem.replace('.tar', '')
        if target_dir.exists():
            continue
        target_dir.mkdir(parents=True, exist_ok=True)
        if archive.suffix == '.zip':
            with zipfile.ZipFile(archive) as zf:
                zf.extractall(target_dir)
        else:
            shutil.unpack_archive(str(archive), str(target_dir))

def materialize_scene(scene_dir: Path, target_root: Path, use_symlink: bool = True) -> Path:
    dst = target_root / scene_dir.name
    if dst.exists():
        return dst
    if use_symlink:
        try:
            os.symlink(scene_dir, dst, target_is_directory=True)
            return dst
        except OSError:
            pass
    shutil.copytree(scene_dir, dst)
    return dst

extract_archives_if_needed(RAW_DATASET_DIR, EXTRACT_ROOT)
scene_dirs = discover_scene_dirs(RAW_DATASET_DIR)
if not scene_dirs:
    scene_dirs = discover_scene_dirs(EXTRACT_ROOT)
if not scene_dirs:
    raise RuntimeError(
        'No ScanNet-style scene folders were found in the attached dataset or extracted archives. '
        'Attach a ScanNet-style subset or edit discover_scene_dirs() once for your layout.'
    )

materialized = [materialize_scene(scene_dir, TARGET_DATA_ROOT) for scene_dir in scene_dirs]
len(materialized), materialized[:5]

In [ ]:
scene_ids = sorted([path.name for path in TARGET_DATA_ROOT.iterdir() if path.is_dir()])
if len(scene_ids) < 2:
    print('Warning: fewer than 2 scenes found. Train/val split will be shallow.')

split_index = max(1, int(0.8 * len(scene_ids)))
train_scenes = scene_ids[:split_index]
val_scenes = scene_ids[split_index:] if split_index < len(scene_ids) else scene_ids[:1]

print('num scenes:', len(scene_ids))
print('train scenes:', train_scenes[:10])
print('val scenes:', val_scenes[:10])

## Download A Depth Anything V2 Checkpoint

For Kaggle, `vits` is the safest first run. You can switch to `vitb` or `vitl` later if the accelerator and runtime budget are sufficient.

In [ ]:
run(
    [
        'python', '-m', 'baseline.depth_anything_v2', 'download',
        '--mode', 'metric',
        '--profile', 'hypersim',
        '--encoder', 'vits',
    ],
    cwd=REPO_DIR,
)

## Build Kaggle-Local Train And Eval Configs

This writes configs into `outputs/kaggle_configs/` so the tracked base configs remain unchanged.

In [ ]:
CONFIG_DIR = REPO_DIR / 'outputs' / 'kaggle_configs'
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

intrinsics = [
    [577.8706, 0.0, 159.5],
    [0.0, 577.8706, 119.5],
    [0.0, 0.0, 1.0],
]

train_config = {
    'seed': 7,
    'device': 'cuda' if shutil.which('nvidia-smi') else 'cpu',
    'output_dir': str(REPO_DIR / 'outputs' / 'phase1_kaggle_train'),
    'data': {
        'root': str(TARGET_DATA_ROOT),
        'image_size': [240, 320],
        'intrinsics': intrinsics,
        'pairing': {
            'min_translation_m': 0.02,
            'max_translation_m': 0.30,
            'min_rotation_deg': 1.0,
            'max_rotation_deg': 10.0,
            'max_index_gap': 30,
        },
        'splits': {
            'train': {'scenes': train_scenes},
            'val': {'scenes': val_scenes},
        },
    },
    'model': {
        'depth': {
            'mode': 'metric',
            'encoder': 'vits',
            'profile': 'hypersim',
            'checkpoint_path': None,
            'freeze_backbone': True,
            'adapter_hidden_channels': 32,
        }
    },
    'train': {
        'batch_size': 2,
        'epochs': 1,
        'lr': 1e-4,
        'weight_decay': 1e-4,
        'log_interval': 10,
        'vis_interval': 50,
        'checkpoint_interval': 100,
    },
    'eval': {'batch_size': 2},
    'losses': {
        'weights': {
            'target_rgb': 1.0,
            'cycle_rgb': 1.0,
            'cycle_depth': 0.5,
            'target_depth': 0.25,
        }
    },
}

eval_config = {
    'device': train_config['device'],
    'data': train_config['data'],
    'model': train_config['model'],
    'eval': {'batch_size': 2},
    'losses': train_config['losses'],
}

train_config_path = CONFIG_DIR / 'train_phase1_kaggle.yaml'
eval_config_path = CONFIG_DIR / 'eval_phase1_kaggle.yaml'
with train_config_path.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(train_config, handle, sort_keys=False)
with eval_config_path.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(eval_config, handle, sort_keys=False)

print(train_config_path)
print(eval_config_path)

## Run Phase 1 Training

The first proper Kaggle run should be small and deliberate. Start with one epoch, confirm outputs and checkpoints, then scale batch size, scenes, or epochs.

In [ ]:
run(
    ['python', '-m', 'eaglevision.cli.train', '--config', str(train_config_path)],
    cwd=REPO_DIR,
)

## Run Baseline Validation Eval

This gives you the frozen baseline reference on the same split.

In [ ]:
baseline_eval = run(
    ['python', '-m', 'eaglevision.cli.eval', '--config', str(eval_config_path), '--baseline-only'],
    cwd=REPO_DIR,
    capture=True,
)

baseline_metrics = {}
for line in baseline_eval.stdout.splitlines():
    if ': ' not in line:
        continue
    key, value = line.split(': ', 1)
    try:
        baseline_metrics[key.strip()] = float(value.strip())
    except ValueError:
        pass

baseline_metrics_path = REPO_DIR / 'outputs' / 'phase1_kaggle_train' / 'baseline_metrics.yaml'
baseline_metrics_path.parent.mkdir(parents=True, exist_ok=True)
with baseline_metrics_path.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(baseline_metrics, handle, sort_keys=False)

baseline_metrics


## Inspect Training Artifacts

You should see JSONL logs, checkpoints, and visualization panels in the run output directory.

## Run Summary

This section creates compact reviewer-friendly artifacts:

- a metrics summary table
- a JSON and CSV summary file
- a zip archive of the training outputs for Kaggle dataset export


In [ ]:
import csv

run_dir = REPO_DIR / 'outputs' / 'phase1_kaggle_train'
summary_dir = run_dir / 'submission_summary'
summary_dir.mkdir(parents=True, exist_ok=True)

summary_rows = []
for metric_name, metric_value in sorted(baseline_metrics.items()):
    summary_rows.append({'split': 'val', 'model': 'baseline', 'metric': metric_name, 'value': metric_value})

summary_json_path = summary_dir / 'summary_metrics.json'
summary_csv_path = summary_dir / 'summary_metrics.csv'
with summary_json_path.open('w', encoding='utf-8') as handle:
    json.dump(summary_rows, handle, indent=2)
with summary_csv_path.open('w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=['split', 'model', 'metric', 'value'])
    writer.writeheader()
    writer.writerows(summary_rows)

summary_rows[:10]


## Package Outputs For Kaggle Export

This creates a single zip archive you can publish as a Kaggle dataset and later mount in the evaluation notebook.


In [ ]:
archive_base = REPO_DIR / 'outputs' / 'phase1_kaggle_train_export'
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=run_dir)
archive_path


In [ ]:
run_dir = REPO_DIR / 'outputs' / 'phase1_kaggle_train'
artifacts = sorted([str(path.relative_to(REPO_DIR)) for path in run_dir.rglob('*')])
artifacts[:100]